# Analisi Competitiva Calzature Outdoor — Scarpa vs Competitor

## Executive Summary

L'analisi ha confrontato **Scarpa** con tre competitor diretti nel segmento calzature outdoor: **Mammut**, **Salewa** e **Dynafit**.

Sono stati analizzati i cataloghi online di ciascun brand su tre segmenti — escursionismo, trail running e lifestyle — raccogliendo nome, descrizione e prezzo di ogni modello tramite web scraping.

**Principali evidenze:**

- Scarpa si posiziona nella **fascia medio-alta** in tutti i segmenti, con un prezzo medio superiore alla media dei competitor nel trail running.
- Scarpa offre una **gamma bilanciata**: 12 modelli trail, 14 escursionismo, 11 lifestyle. Il range di prezzo è ampio soprattutto nel trail (130€ - 260€), a copertura sia del segmento entry level che premium.
- **Mammut** è il competitor con la gamma più ampia (34 modelli solo nell'escursionismo) e prezzi generalmente più accessibili. 
- **Salewa** è il brand più caro nell'escursionismo (media 230€) ma il più economico nel trail e lifestyle. 
- **Dynafit** non presidia il lifestyle — è un brand focalizzato sulla performance. Prezzi competitivi nel trail.

**Raccomandazioni:**

- Valutare un **ampliamento della gamma escursionismo**, dove Mammut ha un vantaggio di ampiezza significativo (34 vs 14 modelli).
- Il segmento **lifestyle** è quello dove Scarpa ha il premium più alto rispetto ai competitor (20€ in più sulla media). 
- Il **trail running** è il segmento più competitivo — Scarpa copre bene il range di prezzo ma potrebbe esplorare modelli entry-level sotto i 130€ per contrastare Salewa e Dynafit.
- Dynafit non ha lifestyle: potrebbe essere un'**opportunità per Scarpa** di conquistare i clienti Dynafit che cercano una scarpa da tutti i giorni restando nel mondo outdoor.

---

## Fase 1 — Selezione dei competitor

Per l'analisi competitiva sono stati selezionati **tre brand** che operano nello stesso segmento di Scarpa, ovvero calzature outdoor/mountain con focus su escursionismo e trail running:

- **Mammut** — brand svizzero con gamma molto ampia, dal trekking al lifestyle. Posizionamento medio.
- **Salewa** — brand altoatesino (gruppo Oberalp), forte nell'escursionismo tecnico. Posizionamento medio-alto.
- **Dynafit** — brand altoatesino (gruppo Oberalp come Salewa), focalizzato su performance e velocità. Non ha segmento lifestyle.

La scelta è ricaduta su brand europei con posizionamento comparabile a Scarpa e con siti e-commerce accessibili via web scraping.
Alcuni brand inizialmente considerati (es. New Balance, La Sportiva) sono stati esclusi perché i loro siti hanno protezioni anti-bot che impediscono la raccolta dati.

I segmenti analizzati per ogni brand sono tre (dove disponibili):

1. **Escursionismo** — scarpe da trekking/hiking
2. **Trail running** — scarpe da corsa su sentiero
3. **Lifestyle** — scarpe urban outdoor / uso quotidiano

Per Dynafit il segmento lifestyle non è disponibile — il brand non lo presidia.

---

## Fase 2 — Raccolta dati (Web Scraping)

I dati sono stati raccolti tramite web scraping dai siti ufficiali dei quattro brand.
Per ciascun brand e segmento sono state estratte tre informazioni: nome del prodotto, descrizione e prezzo.

### 2.1 — Richieste HTTP ai siti

In [14]:
import requests as req

# Scarpa
scarpa_trail_url = "https://scarpa.com/it/collections/scarpe-trail-running-uomo"
scarpa_esc_url = "https://scarpa.com/it/collections/scarpe-da-escursionismo-uomo"
scarpa_life_url = "https://scarpa.com/it/collections/scarpe-urban-outdoor-uomo"

req_scarpa_trail = req.get(scarpa_trail_url)
req_scarpa_esc = req.get(scarpa_esc_url)
req_scarpa_life = req.get(scarpa_life_url)

# Mammut
mammut_trail_url = "https://www.mammut.com/it/it/category/6411/scarpe-da-trail-running"
mammut_esc_url = "https://www.mammut.com/it/it/category/5812/scarpe-da-trekking"
mammut_life_url = "https://www.mammut.com/it/it/category/5814/scarpe-da-tutti-i-giorni"

req_mammut_trail = req.get(mammut_trail_url)
req_mammut_esc = req.get(mammut_esc_url)
req_mammut_life = req.get(mammut_life_url)

# Salewa
salewa_trail_url = "https://www.salewa.com/it-it/uomo-scarpe-speed-hiking"
salewa_esc_url = "https://www.salewa.com/it-it/uomo-scarpe-trekking-hiking"
salewa_life_url = "https://www.salewa.com/it-it/uomo-scarpe-alpine-life"

req_salewa_trail = req.get(salewa_trail_url)
req_salewa_esc = req.get(salewa_esc_url)
req_salewa_life = req.get(salewa_life_url)

# Dynafit (no lifestyle)
dynafit_trail_url = "https://www.dynafit.com/it-it/uomo/calzature/scarpe-running"
dynafit_esc_url = "https://www.dynafit.com/it-it/uomo/calzature/scarpe-alpinismo"

req_dynafit_trail = req.get(dynafit_trail_url)
req_dynafit_esc = req.get(dynafit_esc_url)

# verifica response
print(req_scarpa_trail, req_scarpa_esc, req_scarpa_life)
print(req_mammut_trail, req_mammut_esc, req_mammut_life)
print(req_salewa_trail, req_salewa_esc, req_salewa_life)
print(req_dynafit_trail, req_dynafit_esc)

<Response [200]> <Response [200]> <Response [200]>
<Response [200]> <Response [200]> <Response [200]>
<Response [524]> <Response [200]> <Response [524]>
<Response [200]> <Response [200]>


### 2.2 — Parsing HTML e costruzione DataFrame per brand

Ogni sito ha una struttura HTML diversa, quindi il parsing è stato adattato caso per caso.
I dati estratti sono: nome prodotto, descrizione e prezzo. I duplicati (varianti colore/taglia) sono stati rimossi.

Nota: le classi CSS di Mammut sono particolarmente lunghe e brutte — è la struttura del loro sito, non un errore.

#### Scarpa

In [15]:
from bs4 import BeautifulSoup
import pandas as pd

# parsing scarpa trail
soup_scarpa_trail = BeautifulSoup(req_scarpa_trail.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_scarpa_trail.find_all("a", class_="product-title"):
    nomi.append(n.text.strip())
for d in soup_scarpa_trail.find_all("p", class_="product-card__description"):
    descrizioni.append(d.text.strip())
for p in soup_scarpa_trail.find_all("sale-price"):
    prezzi.append(p.text.strip())
scarpa_trail = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
scarpa_trail["prezzo"] = scarpa_trail["prezzo"].str.replace("Prezzo scontato", "")
scarpa_trail["prezzo"] = scarpa_trail["prezzo"].str.replace("A partire da ", "")
scarpa_trail["prezzo"] = scarpa_trail["prezzo"].str.replace("€", "")
scarpa_trail["prezzo"] = scarpa_trail["prezzo"].str.replace(",", ".")
scarpa_trail["prezzo"] = scarpa_trail["prezzo"].astype(float)
scarpa_trail = scarpa_trail.drop_duplicates(subset="nome")

# escursionismo
soup_scarpa_esc = BeautifulSoup(req_scarpa_esc.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_scarpa_esc.find_all("a", class_="product-title"):
    nomi.append(n.text.strip())
for d in soup_scarpa_esc.find_all("p", class_="product-card__description"):
    descrizioni.append(d.text.strip())
for p in soup_scarpa_esc.find_all("sale-price"):
    prezzi.append(p.text.strip())
scarpa_esc = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
scarpa_esc["prezzo"] = scarpa_esc["prezzo"].str.replace("Prezzo scontato", "")
scarpa_esc["prezzo"] = scarpa_esc["prezzo"].str.replace("A partire da ", "")
scarpa_esc["prezzo"] = scarpa_esc["prezzo"].str.replace("€", "")
scarpa_esc["prezzo"] = scarpa_esc["prezzo"].str.replace(",", ".")
scarpa_esc["prezzo"] = scarpa_esc["prezzo"].astype(float)
scarpa_esc = scarpa_esc.drop_duplicates(subset="nome")

# lifestyle
soup_scarpa_life = BeautifulSoup(req_scarpa_life.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_scarpa_life.find_all("a", class_="product-title"):
    nomi.append(n.text.strip())
for d in soup_scarpa_life.find_all("p", class_="product-card__description"):
    descrizioni.append(d.text.strip())
for p in soup_scarpa_life.find_all("sale-price"):
    prezzi.append(p.text.strip())
scarpa_life = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
scarpa_life["prezzo"] = scarpa_life["prezzo"].str.replace("Prezzo scontato", "")
scarpa_life["prezzo"] = scarpa_life["prezzo"].str.replace("A partire da ", "")
scarpa_life["prezzo"] = scarpa_life["prezzo"].str.replace("€", "")
scarpa_life["prezzo"] = scarpa_life["prezzo"].str.replace(",", ".")
scarpa_life["prezzo"] = scarpa_life["prezzo"].astype(float)
scarpa_life = scarpa_life.drop_duplicates(subset="nome")


#### Mammut

In [16]:
# trail
soup_mammut_trail = BeautifulSoup(req_mammut_trail.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_mammut_trail.find_all("div", class_="ProductCardInfo-module-scss-module__sCZr7G__productNameSection"):
    nomi.append(n.text.strip())
for d in soup_mammut_trail.find_all("div", class_="ProductCardInfo-module-scss-module__sCZr7G__productDescription"):
    descrizioni.append(d.text.strip())
for p in soup_mammut_trail.find_all("div", class_="PriceDisplay-module-scss-module__gzqClq__container"):
    prezzi.append(p.find("span", attrs={"aria-hidden": "true"}).text.strip())
mammut_trail = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
mammut_trail["prezzo"] = mammut_trail["prezzo"].str.replace("€", "")
mammut_trail["prezzo"] = mammut_trail["prezzo"].str.replace(",", ".")
mammut_trail["prezzo"] = mammut_trail["prezzo"].astype(float)
mammut_trail = mammut_trail.drop_duplicates(subset="nome")

# escursionismo
soup_mammut_esc = BeautifulSoup(req_mammut_esc.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_mammut_esc.find_all("div", class_="ProductCardInfo-module-scss-module__sCZr7G__productNameSection"):
    nomi.append(n.text.strip())
for d in soup_mammut_esc.find_all("div", class_="ProductCardInfo-module-scss-module__sCZr7G__productDescription"):
    descrizioni.append(d.text.strip())
for p in soup_mammut_esc.find_all("div", class_="PriceDisplay-module-scss-module__gzqClq__container"):
    prezzi.append(p.find("span", attrs={"aria-hidden": "true"}).text.strip())
mammut_esc = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
mammut_esc["prezzo"] = mammut_esc["prezzo"].str.replace("€", "")
mammut_esc["prezzo"] = mammut_esc["prezzo"].str.replace(",", ".")
mammut_esc["prezzo"] = mammut_esc["prezzo"].astype(float)
mammut_esc = mammut_esc.drop_duplicates(subset="nome")

# lifestyle
soup_mammut_life = BeautifulSoup(req_mammut_life.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_mammut_life.find_all("div", class_="ProductCardInfo-module-scss-module__sCZr7G__productNameSection"):
    nomi.append(n.text.strip())
for d in soup_mammut_life.find_all("div", class_="ProductCardInfo-module-scss-module__sCZr7G__productDescription"):
    descrizioni.append(d.text.strip())
for p in soup_mammut_life.find_all("div", class_="PriceDisplay-module-scss-module__gzqClq__container"):
    prezzi.append(p.find("span", attrs={"aria-hidden": "true"}).text.strip())
mammut_life = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
mammut_life["prezzo"] = mammut_life["prezzo"].str.replace("€", "")
mammut_life["prezzo"] = mammut_life["prezzo"].str.replace(",", ".")
mammut_life["prezzo"] = mammut_life["prezzo"].astype(float)
mammut_life = mammut_life.drop_duplicates(subset="nome")

print("MAMMUT trail:", mammut_trail.shape)
print("MAMMUT esc:", mammut_esc.shape)
print("MAMMUT life:", mammut_life.shape)

MAMMUT trail: (12, 3)
MAMMUT esc: (34, 3)
MAMMUT life: (27, 3)


#### Salewa

In [17]:
# trail
soup_salewa_trail = BeautifulSoup(req_salewa_trail.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_salewa_trail.find_all("p", attrs={"data-testid": "product-name"}):
    nomi.append(n.text.strip())
for d in soup_salewa_trail.find_all("span", class_="truncated"):
    descrizioni.append(d.text.strip())
for p in soup_salewa_trail.find_all("span", attrs={"data-testid": "regular-price"}):
    prezzi.append(p.text.strip())
salewa_trail = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
salewa_trail["prezzo"] = salewa_trail["prezzo"].str.replace("\xa0€", "")
salewa_trail["prezzo"] = salewa_trail["prezzo"].str.replace(",", ".")
salewa_trail["prezzo"] = salewa_trail["prezzo"].astype(float)
salewa_trail = salewa_trail.drop_duplicates(subset="nome")

# escursionismo
soup_salewa_esc = BeautifulSoup(req_salewa_esc.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_salewa_esc.find_all("p", attrs={"data-testid": "product-name"}):
    nomi.append(n.text.strip())
for d in soup_salewa_esc.find_all("span", class_="truncated"):
    descrizioni.append(d.text.strip())
for p in soup_salewa_esc.find_all("span", attrs={"data-testid": "regular-price"}):
    prezzi.append(p.text.strip())
salewa_esc = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
salewa_esc["prezzo"] = salewa_esc["prezzo"].str.replace("\xa0€", "")
salewa_esc["prezzo"] = salewa_esc["prezzo"].str.replace(",", ".")
salewa_esc["prezzo"] = salewa_esc["prezzo"].astype(float)
salewa_esc = salewa_esc.drop_duplicates(subset="nome")

# lifestyle
soup_salewa_life = BeautifulSoup(req_salewa_life.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_salewa_life.find_all("p", attrs={"data-testid": "product-name"}):
    nomi.append(n.text.strip())
for d in soup_salewa_life.find_all("span", class_="truncated"):
    descrizioni.append(d.text.strip())
for p in soup_salewa_life.find_all("span", attrs={"data-testid": "regular-price"}):
    prezzi.append(p.text.strip())
salewa_life = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
salewa_life["prezzo"] = salewa_life["prezzo"].str.replace("\xa0€", "")
salewa_life["prezzo"] = salewa_life["prezzo"].str.replace(",", ".")
salewa_life["prezzo"] = salewa_life["prezzo"].astype(float)
salewa_life = salewa_life.drop_duplicates(subset="nome")

print("SALEWA trail:", salewa_trail.shape)
print("SALEWA esc:", salewa_esc.shape)
print("SALEWA life:", salewa_life.shape)

AttributeError: Can only use .str accessor with string values!

#### Dynafit
Dynafit non ha il segmento lifestyle — è un brand focalizzato su performance e gare.

In [ ]:
# trail
soup_dynafit_trail = BeautifulSoup(req_dynafit_trail.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_dynafit_trail.find_all("p", attrs={"data-testid": "product-name"}):
    nomi.append(n.text.strip())
for d in soup_dynafit_trail.find_all("span", class_="truncated"):
    descrizioni.append(d.text.strip())
for p in soup_dynafit_trail.find_all("span", attrs={"data-testid": "regular-price"}):
    prezzi.append(p.text.strip())
dynafit_trail = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
dynafit_trail["prezzo"] = dynafit_trail["prezzo"].str.replace("\xa0€", "")
dynafit_trail["prezzo"] = dynafit_trail["prezzo"].str.replace(",", ".")
dynafit_trail["prezzo"] = dynafit_trail["prezzo"].astype(float)
dynafit_trail = dynafit_trail.drop_duplicates(subset="nome")

# escursionismo
soup_dynafit_esc = BeautifulSoup(req_dynafit_esc.text)
nomi = [] 
descrizioni = []
prezzi = []
for n in soup_dynafit_esc.find_all("p", attrs={"data-testid": "product-name"}):
    nomi.append(n.text.strip())
for d in soup_dynafit_esc.find_all("span", class_="truncated"):
    descrizioni.append(d.text.strip())
for p in soup_dynafit_esc.find_all("span", attrs={"data-testid": "regular-price"}):
    prezzi.append(p.text.strip())
dynafit_esc = pd.DataFrame({"nome": nomi, "descrizione": descrizioni, "prezzo": prezzi})
dynafit_esc["prezzo"] = dynafit_esc["prezzo"].str.replace("\xa0€", "")
dynafit_esc["prezzo"] = dynafit_esc["prezzo"].str.replace(",", ".")
dynafit_esc["prezzo"] = dynafit_esc["prezzo"].astype(float)
dynafit_esc = dynafit_esc.drop_duplicates(subset="nome")

print("DYNAFIT trail:", dynafit_trail.shape)
print("DYNAFIT esc:", dynafit_esc.shape)

---

## Fase 3 — Elaborazione e analisi

### 3.1 — Dataset unificato

Aggiungiamo le colonne brand e segmento e mettiamo tutto insieme.

In [ ]:
# aggiungiamo brand e segmento
scarpa_trail["brand"] = "Scarpa"
scarpa_trail["segmento"] = "trail"
scarpa_esc["brand"] = "Scarpa"
scarpa_esc["segmento"] = "escursionismo"
scarpa_life["brand"] = "Scarpa"
scarpa_life["segmento"] = "lifestyle"

mammut_trail["brand"] = "Mammut"
mammut_trail["segmento"] = "trail"
mammut_esc["brand"] = "Mammut"
mammut_esc["segmento"] = "escursionismo"
mammut_life["brand"] = "Mammut"
mammut_life["segmento"] = "lifestyle"

salewa_trail["brand"] = "Salewa"
salewa_trail["segmento"] = "trail"
salewa_esc["brand"] = "Salewa"
salewa_esc["segmento"] = "escursionismo"
salewa_life["brand"] = "Salewa"
salewa_life["segmento"] = "lifestyle"

dynafit_trail["brand"] = "Dynafit"
dynafit_trail["segmento"] = "trail"
dynafit_esc["brand"] = "Dynafit"
dynafit_esc["segmento"] = "escursionismo"

df = pd.concat([
    scarpa_trail, scarpa_esc, scarpa_life,
    mammut_trail, mammut_esc, mammut_life,
    salewa_trail, salewa_esc, salewa_life,
    dynafit_trail, dynafit_esc
])

print(df.shape)

### 3.2 — Quanti modelli ha ogni brand per segmento?

In [ ]:
import matplotlib.pyplot as plt

ordine_brand = ["Scarpa", "Mammut", "Salewa", "Dynafit"]
ordine_segmento = ["escursionismo", "trail", "lifestyle"]

# conteggio
modelli = df.groupby(["brand", "segmento"])["nome"].count()
print(modelli)

plt.figure()
modelli.unstack().reindex(ordine_brand)[ordine_segmento].plot(kind="bar", figsize=(10, 6))
plt.title("Numero di modelli per brand e segmento")
plt.xlabel("Brand")
plt.ylabel("Numero di modelli")
plt.legend(title="Segmento")
plt.show()

Mammut ha una gamma molto più ampia degli altri, soprattutto nell'escursionismo (34 modelli!). Scarpa è ben bilanciata tra i tre segmenti.

### 3.3 — Prezzi medi a confronto

In [ ]:
# prezzi per brand e segmento
prezzi_analisi = df.groupby(["brand", "segmento"])["prezzo"].agg(["mean", "min", "max"])
print(prezzi_analisi)

plt.figure()
prezzi_analisi["mean"].unstack().reindex(ordine_brand)[ordine_segmento].plot(kind="bar", figsize=(10, 6))
plt.title("Prezzo medio per brand e segmento")
plt.xlabel("Brand")
plt.ylabel("Prezzo medio (€)")
plt.legend(title="Segmento")
plt.show()

L'escursionismo è il segmento più caro per tutti. Salewa ha una forbice notevole tra escursionismo e trail/lifestyle.

### 3.4 — Scarpa vs media competitor

In [ ]:
scarpa_mean = df[df["brand"] == "Scarpa"].groupby("segmento")["prezzo"].mean()
media_competitor = df[df["brand"] != "Scarpa"].groupby("segmento")["prezzo"].mean()

confronto = pd.DataFrame({
    "Scarpa": scarpa_mean,
    "Media competitor": media_competitor
})

# verde per scarpa, grigio per i competitor
plt.figure()
confronto.reindex(ordine_segmento).plot(kind="bar", figsize=(10, 6), color=["green", "gray"])
plt.title("Scarpa vs Media Competitor per segmento")
plt.xlabel("Segmento")
plt.ylabel("Prezzo medio (€)")
plt.legend(title="Brand")
plt.show()

Scarpa è sopra la media competitor in tutti i segmenti, ma il gap più evidente è nel lifestyle.

### 3.5 — Distribuzione prezzi (boxplot)

In [ ]:
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="brand", y="prezzo", hue="segmento",
            order=ordine_brand, hue_order=ordine_segmento)
plt.title("Distribuzione prezzi per brand e segmento")
plt.xlabel("Brand")
plt.ylabel("Prezzo (€)")
plt.legend(title="Segmento")
plt.show()

Il boxplot conferma che Scarpa ha una distribuzione ampia nel trail (copre bene entry level e premium). Mammut ha molti outlier, segno di una gamma eterogenea. Dynafit è il più compatto — poca variazione di prezzo.

### 3.6 — Tabella riassuntiva

In [ ]:
tabella = df.groupby(["brand", "segmento"])["prezzo"].agg(["min", "mean", "max", "count"])
tabella.columns = ["Prezzo Min", "Prezzo Medio", "Prezzo Max", "N. Modelli"]
tabella["Prezzo Medio"] = tabella["Prezzo Medio"].round(2)
tabella

---

## Conclusione

L'analisi ha permesso di mappare il posizionamento competitivo di Scarpa su tre segmenti.

**In sintesi:** Scarpa si posiziona come brand premium con gamma equilibrata. Il principale vantaggio competitivo è la copertura ampia del range di prezzo nel trail running. L'area di attenzione principale è l'ampiezza della gamma escursionismo, dove Mammut ha un vantaggio significativo.

**Possibili sviluppi futuri:**

- Analisi della varietà di taglie e colori, entrando nelle pagine dei singoli prodotti
- Monitoraggio dei prezzi nel tempo per capire le strategie promozionali dei competitor
- Estensione dell'analisi ad altri brand (Salomon, Merrell, Hoka) e ad altri segmenti (donna, bambino)
- Integrazione con dati di vendita interni per verificare se il posizionamento di prezzo corrisponde ai volumi effettivi